# IoU and non-maximum suppression

Implement two calculations used in object detection: overlap between boxes and suppression of redundant predictions. This lesson uses handcrafted boxes and scores; it does not train or evaluate a detector.

**Prerequisites:** NumPy arrays. **Environment:** base requirements. **Execution:** restart and run all cells. No model weights, images, Ultralytics, or GPU are required.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def boxes_array(boxes):
    boxes = np.asarray(boxes, dtype=float).reshape(-1, 4)
    if not np.isfinite(boxes).all() or np.any(boxes[:, 2:] < boxes[:, :2]):
        raise ValueError('Boxes must be finite xyxy coordinates with nonnegative width/height')
    return boxes

def iou_one_to_many(box, boxes):
    box, boxes = boxes_array([box])[0], boxes_array(boxes)
    top_left = np.maximum(box[:2], boxes[:, :2])
    bottom_right = np.minimum(box[2:], boxes[:, 2:])
    intersection = np.prod(np.maximum(bottom_right - top_left, 0), axis=1)
    area_box = np.prod(box[2:] - box[:2])
    area_boxes = np.prod(boxes[:, 2:] - boxes[:, :2], axis=1)
    union = area_box + area_boxes - intersection
    return np.divide(intersection, union, out=np.zeros_like(union), where=union > 0)


## Define coordinates and check a hand calculation

We use continuous `(x_min, y_min, x_max, y_max)` coordinates and areas `(x_max-x_min)*(y_max-y_min)`, with no pixel-inclusive `+1`. IoU is intersection area divided by union area. The boxes `[0,0,2,2]` and `[1,1,3,3]` intersect over area 1 and have union area 7, giving IoU $1/7$. A zero-area union is defined here to have IoU zero.


In [ ]:
np.testing.assert_allclose(iou_one_to_many([0, 0, 2, 2], [[1, 1, 3, 3]]), [1 / 7])
np.testing.assert_allclose(iou_one_to_many([0, 0, 2, 2], [[0, 0, 2, 2], [3, 3, 5, 5]]), [1, 0])
np.testing.assert_allclose(iou_one_to_many([0, 0, 0, 0], [[0, 0, 0, 0]]), [0])
assert iou_one_to_many([0, 0, 1, 1], []).size == 0
print('Hand-calculated overlap and edge cases pass')


## Keep high-scoring boxes, suppress overlapping boxes of the same class

Process scores in descending order, breaking ties by original index. After keeping one box, remove lower-scored boxes of the same class when IoU exceeds the threshold. Different classes remain independent. Confidence filtering is a separate operation; NMS alone does not determine detector quality.


In [ ]:
def class_aware_nms(boxes, scores, classes, threshold=0.5):
    boxes = boxes_array(boxes)
    scores, classes = np.asarray(scores, dtype=float), np.asarray(classes)
    if scores.shape != (len(boxes),) or classes.shape != (len(boxes),):
        raise ValueError('Scores and classes must have one entry per box')
    if not np.isfinite(scores).all() or not 0 <= threshold <= 1:
        raise ValueError('Finite scores and an IoU threshold in [0, 1] are required')
    order, kept = np.argsort(-scores, kind='stable'), []
    while order.size:
        current, rest = order[0], order[1:]
        kept.append(int(current))
        overlap = iou_one_to_many(boxes[current], boxes[rest])
        suppress = (classes[rest] == classes[current]) & (overlap > threshold)
        order = rest[~suppress]
    return np.array(kept, dtype=int)

boxes = np.array([[0, 0, 2, 2], [0.1, 0.1, 2.1, 2.1], [3, 3, 5, 5],
                  [0, 0, 2, 2], [3.1, 3.1, 5.1, 5.1]])
scores = np.array([0.95, 0.9, 0.8, 0.85, 0.7])
classes = np.array([0, 0, 0, 1, 0])
kept = class_aware_nms(boxes, scores, classes)
np.testing.assert_array_equal(kept, [0, 3, 2])
np.testing.assert_array_equal(class_aware_nms([], [], []), [])
print('Kept original indices:', kept.tolist())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, indices, title in zip(axes, [np.arange(len(boxes)), kept], ['Before NMS', 'After class-aware NMS']):
    for index in indices:
        x1, y1, x2, y2 = boxes[index]
        color = ['tab:blue', 'tab:orange'][classes[index]]
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color,
                               linewidth=2, linestyle='--' if classes[index] else '-'))
        ax.text(x1, y1 + 0.18 * index, f'#{index}: {scores[index]:.2f}', color=color)
    ax.set(xlim=(-0.4, 5.6), ylim=(5.6, -0.4), title=title, xlabel='x', ylabel='y')
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()


## Practice

Change the threshold and explain which boxes survive. Compare class-aware and class-agnostic suppression. Continue with the [YOLO project](../../../projects/object_detection/yolo/README.md) when you have the required model and dataset; detector metrics such as mAP require labeled images and are not measured here.

**Reference:** [torchvision NMS](https://docs.pytorch.org/vision/stable/generated/torchvision.ops.nms.html). The implementation here deliberately provides stable tie behavior for the lesson.
